In [ ]:
!pip install qibo

In [ ]:
!pip install qibojit

In [ ]:
import qibo
qibo.set_backend("qibojit")

https://qibo.science/qibo/stable/code-examples/tutorials/aavqe/README.html

¿Cuánta profundidad (layers) necesita un circuito variacional para aproximar el estado fundamental de un sistema físico? Ejemplo: modelo de Ising

statevector y algoritmos variacionales

La profundidad del circuito controla el entanglement que puedes representar:

pocas capas → poco entrelazamiento → mala aproximación

muchas capas → más expresividad → mejor estado

Preparar el ground state usando circuitos tipo VQE / QAOA, variando N, p, tetha y estudiar Error=E(θ)−E0

Notice that any potential advantage of the VQE could be lost without practical approaches to perform the parameter optimization due to the high-dimensional parameter landscape and the problem of barren plateaus.

Solución: AAVQE = Adiabatically Assisted VQE: Empiezas con un Hamiltoniano fácil: H0 (ground state trivial) y vas deformando poco a poco: s:0→1
En cada paso: optimizas VQE y usas parámetros anteriores como inicialización

H(s)=(1−s)H0​+sHP​

En lugar de optimizar directamente para HP:

1.Empiezas en s=0 → sistema fácil

2.Optimiza el circuito variacional

3.Aumentas s

4.Reutilizas los parámetros anteriores

5.Repites

Evita:

- mínimos locales

- barren plateaus

In [ ]:
import argparse

import numpy as np

from qibo import Circuit, gates
from qibo.hamiltonians import XXZ, X
from qibo.models.variational import AAVQE
from qibo.hamiltonians import TFIM

#Ansatz (circuito variacional)
def main(nqubits, layers, maxsteps, T_max):
    circuit = Circuit(nqubits)
    for l in range(layers): #un circuito tipo Trotter
        circuit.add(gates.RY(q, theta=0) for q in range(nqubits))#RY → parámetros variacionales
        circuit.add(gates.CZ(q, q + 1) for q in range(0, nqubits - 1, 2))#CZ → genera entrelazamiento local
        circuit.add(gates.RY(q, theta=0) for q in range(nqubits))
        circuit.add(gates.CZ(q, q + 1) for q in range(1, nqubits - 2, 2))
        circuit.add(gates.CZ(0, nqubits - 1))
    circuit.add(gates.RY(q, theta=0) for q in range(nqubits))

    problem_hamiltonian = TFIM(nqubits, h=1.0)
    easy_hamiltonian = X(nqubits) #estado trivial (todos en |+⟩)
    s = lambda t: t
    #AAVQE
    aavqe = AAVQE(
        circuit, easy_hamiltonian, problem_hamiltonian, s, nsteps=maxsteps, t_max=T_max
    )#nsteps=teraciones por cada valor de s,T_max=número de pasos adiabáticos(discretización de s)
    #Inicialización
    initial_parameters = np.random.uniform(
        0, 2 * np.pi * 0.1, 2 * nqubits * layers + nqubits
    )
    best, params = aavqe.minimize(initial_parameters) #Optimización, optimización progresiva (mucho más estable) que QAOA

    print("Final parameters: ", params)
    print("Final energy: ", best)

    # We compute the difference from the exact value to check performance
    eigenvalue = problem_hamiltonian.eigenvalues() #diagonaliza exactamente (solo posible para N pequeño
    print(eigenvalue)
    print("Difference from exact value: ", best - np.real(eigenvalue[0]))
    print("Log difference: ", -np.log10(best - np.real(eigenvalue[0])))

    exact = np.real(eigenvalue[0])
    error = abs(best - exact)

    return best, exact

def experiment():

    nqubits = 8
    layers_list = [1, 2, 3, 4, 5, 6]

    results = []

    for layers in layers_list:
        best, exact = main(
            nqubits=nqubits,
            layers=layers,
            maxsteps=200,
            T_max=10
        )

        error = abs(best - exact)
        results.append((layers, error))

    print("\n===== ESCALADO =====")
    for l, e in results:
        print(f"layers={l}, error={e}")

In [ ]:
if __name__ == "__main__":
    experiment()

Final parameters:  [-0.34837183  0.24351205  0.42168766 -0.37054101 -0.40434396  0.21077554
 -0.38086424  0.41012642 -0.09413715 -0.66244414 -0.09686169  0.63983712
 -0.03671193 -0.62726291  0.6511163  -0.08441653  1.0089382   1.00896851
  0.32989387  0.32988886  1.00895473  1.00897295  0.32989978  0.32987514]
Final energy:  -10.11743157770502
[-1.02516618e+01 -1.00546790e+01 -8.69093921e+00 -8.52394525e+00
 -8.52394525e+00 -7.24901957e+00 -7.24901957e+00 -7.24901957e+00
 -7.24901957e+00 -7.22625186e+00 -7.22625186e+00 -6.99321153e+00
 -6.35916085e+00 -6.35916085e+00 -6.14542205e+00 -6.14542205e+00
 -6.14542205e+00 -6.14542205e+00 -6.05467898e+00 -5.80709993e+00
 -5.69551813e+00 -5.69551813e+00 -5.69551813e+00 -5.69551813e+00
 -5.54815938e+00 -5.54815938e+00 -5.54815938e+00 -5.54815938e+00
 -4.82842712e+00 -4.82842712e+00 -4.82842712e+00 -4.82842712e+00
 -4.70350241e+00 -4.70350241e+00 -4.70350241e+00 -4.70350241e+00
 -4.52394525e+00 -4.52394525e+00 -4.39782473e+00 -4.24637735e+00
 -4.